In [0]:
-- Sanity check compatível com Serverless
SELECT current_catalog(), current_database(), current_timestamp();

current_catalog(),current_database(),current_timestamp()
workspace,default,2026-08-24T18:37:05.367Z


In [0]:
-- Valida acesso aos dados nativos
SHOW TABLES IN samples.tpch;

database,tableName,isTemporary
tpch,customer,false
tpch,lineitem,false
tpch,nation,false
tpch,orders,false
tpch,part,false
tpch,partsupp,false
tpch,region,false
tpch,supplier,false


In [0]:
-- Cria o banco de dados do projeto (isolado do default)
CREATE DATABASE IF NOT EXISTS ruptura_rj
COMMENT 'Lakehouse: Detecção de Ruptura de Estoque - Varejo RJ';

USE ruptura_rj;

In [0]:
CREATE OR REPLACE TABLE ruptura_rj.bronze_orders
USING DELTA
COMMENT 'Bronze: pedidos de venda brutos - fonte samples.tpch'
AS
SELECT
    o_orderkey        AS order_id,
    o_custkey         AS customer_id,
    o_orderstatus     AS order_status,
    o_totalprice      AS total_price,
    o_orderdate       AS order_date,
    o_orderpriority   AS order_priority,
    o_shippriority    AS ship_priority,
    o_comment         AS raw_comment,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.orders;

-- Valida contagem
SELECT COUNT(*) AS total_pedidos FROM ruptura_rj.bronze_orders;

total_pedidos
7500000


In [0]:
CREATE OR REPLACE TABLE ruptura_rj.bronze_lineitem
USING DELTA
COMMENT 'Bronze: itens de pedido brutos - fonte samples.tpch'
AS
SELECT
    l_orderkey        AS order_id,
    l_partkey         AS product_id,
    l_suppkey         AS supplier_id,
    l_linenumber      AS line_number,
    l_quantity        AS quantity,
    l_extendedprice   AS gross_revenue,
    l_discount        AS discount_pct,
    l_tax             AS tax_pct,
    l_returnflag      AS return_flag,
    l_linestatus      AS line_status,
    l_shipdate        AS ship_date,
    l_commitdate      AS commit_date,
    l_receiptdate     AS receipt_date,
    l_shipmode        AS ship_mode,
    l_comment         AS raw_comment,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.lineitem;

SELECT COUNT(*) AS total_itens FROM ruptura_rj.bronze_lineitem;

total_itens
29999795


In [0]:
-- Produtos (SKUs)
CREATE OR REPLACE TABLE ruptura_rj.bronze_parts
USING DELTA
COMMENT 'Bronze: catálogo de produtos - fonte samples.tpch'
AS
SELECT
    p_partkey    AS product_id,
    p_name       AS product_name,
    p_mfgr       AS manufacturer,
    p_brand      AS brand,
    p_type       AS product_type,
    p_size       AS size_info,
    p_container  AS container_type,
    p_retailprice AS retail_price,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.part;

-- Fornecedores
CREATE OR REPLACE TABLE ruptura_rj.bronze_supplier
USING DELTA
AS
SELECT
    s_suppkey   AS supplier_id,
    s_name      AS supplier_name,
    s_address   AS address,
    s_nationkey AS nation_id,
    s_phone     AS phone,
    s_acctbal   AS account_balance,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.supplier;

-- Contratos ERP: produto x fornecedor (lead time e custo)
CREATE OR REPLACE TABLE ruptura_rj.bronze_partsupp
USING DELTA
AS
SELECT
    ps_partkey    AS product_id,
    ps_suppkey    AS supplier_id,
    ps_availqty   AS available_qty,
    ps_supplycost AS supply_cost,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.partsupp;

-- Clientes
CREATE OR REPLACE TABLE ruptura_rj.bronze_customer
USING DELTA
AS
SELECT
    c_custkey    AS customer_id,
    c_name       AS customer_name,
    c_nationkey  AS nation_id,
    c_mktsegment AS market_segment,
    c_acctbal    AS account_balance,
    current_timestamp() AS ingestion_ts
FROM samples.tpch.customer;

SELECT 'bronze_parts'    AS tabela, COUNT(*) AS registros FROM ruptura_rj.bronze_parts    UNION ALL
SELECT 'bronze_supplier' AS tabela, COUNT(*) AS registros FROM ruptura_rj.bronze_supplier  UNION ALL
SELECT 'bronze_partsupp' AS tabela, COUNT(*) AS registros FROM ruptura_rj.bronze_partsupp  UNION ALL
SELECT 'bronze_customer' AS tabela, COUNT(*) AS registros FROM ruptura_rj.bronze_customer;

tabela,registros
bronze_parts,1000000
bronze_supplier,50000
bronze_partsupp,4000000
bronze_customer,750000


In [0]:
CREATE OR REPLACE TABLE ruptura_rj.silver_itens
USING DELTA
COMMENT 'Silver: itens limpos com receita líquida calculada e flags de qualidade'
AS
WITH dedup_itens AS (
    -- Remove duplicatas estruturais (order_id + product_id + line_number)
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id, product_id, line_number
               ORDER BY ingestion_ts DESC
           ) AS rn
    FROM ruptura_rj.bronze_lineitem
),
itens_validos AS (
    SELECT
        order_id,
        product_id,
        supplier_id,
        line_number,
        quantity,
        gross_revenue,
        discount_pct,
        tax_pct,
        return_flag,
        line_status,
        ship_date,
        commit_date,
        receipt_date,
        ship_mode,
        -- Receita líquida real
        ROUND(gross_revenue * (1 - discount_pct) * (1 + tax_pct), 2) AS net_revenue,
        -- Dias de atraso na entrega (simula ruptura logística)
        DATEDIFF(receipt_date, commit_date)                           AS delivery_delay_days,
        -- Flag de devolução
        CASE WHEN return_flag = 'R' THEN TRUE ELSE FALSE END          AS is_returned,
        -- Flag de entrega atrasada (proxy de ruptura logística)
        CASE
            WHEN DATEDIFF(receipt_date, commit_date) > 0 THEN TRUE
            ELSE FALSE
        END AS is_late_delivery,
        -- Extrai ano e mês para particionamento analítico
        YEAR(ship_date)  AS ship_year,
        MONTH(ship_date) AS ship_month
    FROM dedup_itens
    WHERE rn = 1
      AND quantity       IS NOT NULL AND quantity > 0       -- sem qtd nula
      AND gross_revenue  IS NOT NULL AND gross_revenue > 0  -- sem receita nula
      AND ship_date      IS NOT NULL                        -- sem data nula
      AND product_id     IS NOT NULL
      AND supplier_id    IS NOT NULL
)
SELECT * FROM itens_validos;

-- Relatório de qualidade pós-limpeza
SELECT
    COUNT(*)                                           AS total_registros,
    SUM(CASE WHEN is_returned      THEN 1 ELSE 0 END) AS total_devolvidos,
    SUM(CASE WHEN is_late_delivery THEN 1 ELSE 0 END) AS total_atrasados,
    ROUND(AVG(delivery_delay_days), 1)                AS media_atraso_dias,
    ROUND(SUM(net_revenue), 2)                        AS receita_liquida_total
FROM ruptura_rj.silver_itens;

total_registros,total_devolvidos,total_atrasados,media_atraso_dias,receita_liquida_total
29999795,7406353,18968104,16.5,1133439473148.28


In [0]:
CREATE OR REPLACE TABLE ruptura_rj.silver_risco_ruptura
USING DELTA
COMMENT 'Silver: análise de risco de ruptura por produto e fornecedor'
AS
WITH demanda_por_produto AS (
    -- Demanda total histórica por produto
    SELECT
        product_id,
        COUNT(DISTINCT order_id)        AS total_pedidos,
        SUM(quantity)                   AS demanda_total_qty,
        ROUND(SUM(net_revenue), 2)      AS receita_total,
        ROUND(AVG(delivery_delay_days), 2) AS media_atraso_entrega,
        SUM(CASE WHEN is_late_delivery THEN 1 ELSE 0 END) AS qtd_entregas_atrasadas,
        SUM(CASE WHEN is_returned      THEN 1 ELSE 0 END) AS qtd_devolvidos,
        MAX(ship_date)                  AS ultima_venda
    FROM ruptura_rj.silver_itens
    GROUP BY product_id
),
estoque_disponivel AS (
    -- Estoque disponível no contrato ERP (agregado por produto, somando todos fornecedores)
    SELECT
        product_id,
        SUM(available_qty)              AS estoque_total_disponivel,
        MIN(supply_cost)                AS menor_custo_reposicao,
        COUNT(DISTINCT supplier_id)     AS num_fornecedores
    FROM ruptura_rj.bronze_partsupp
    GROUP BY product_id
),
analise_ruptura AS (
    SELECT
        d.product_id,
        p.product_name,
        p.brand,
        p.product_type,
        p.retail_price,
        d.total_pedidos,
        d.demanda_total_qty,
        d.receita_total,
        d.media_atraso_entrega,
        d.qtd_entregas_atrasadas,
        d.qtd_devolvidos,
        d.ultima_venda,
        e.estoque_total_disponivel,
        e.menor_custo_reposicao,
        e.num_fornecedores,
        -- Cobertura de estoque: quantos dias de estoque sobrando vs demanda diária
        ROUND(
            e.estoque_total_disponivel / NULLIF((d.demanda_total_qty / 365.0), 0),
        1) AS cobertura_estoque_dias,
        -- Receita em risco: se estoque < demanda, qual a perda potencial
        CASE
            WHEN e.estoque_total_disponivel < d.demanda_total_qty
            THEN ROUND((d.demanda_total_qty - e.estoque_total_disponivel)
                       * (d.receita_total / NULLIF(d.demanda_total_qty, 0)), 2)
            ELSE 0
        END AS receita_em_risco,
        -- Score de risco composto (0 a 100)
        ROUND(
            LEAST(100,
                (CASE WHEN e.estoque_total_disponivel < d.demanda_total_qty THEN 40 ELSE 0 END)
              + (LEAST(30, d.media_atraso_entrega * 3))
              + (LEAST(20, (d.qtd_entregas_atrasadas * 100.0
                            / NULLIF(d.total_pedidos, 0))))
              + (CASE WHEN e.num_fornecedores = 1 THEN 10 ELSE 0 END)
            ), 1
        ) AS score_risco_ruptura,
        -- Classificação de risco
        CASE
            WHEN e.estoque_total_disponivel < d.demanda_total_qty
              OR d.media_atraso_entrega > 5  THEN 'CRITICO'
            WHEN d.media_atraso_entrega > 2
              OR (d.qtd_entregas_atrasadas * 100.0
                  / NULLIF(d.total_pedidos, 0)) > 30 THEN 'ALTO'
            WHEN (d.qtd_entregas_atrasadas * 100.0
                  / NULLIF(d.total_pedidos, 0)) > 10 THEN 'MEDIO'
            ELSE 'BAIXO'
        END AS nivel_risco
    FROM demanda_por_produto d
    JOIN estoque_disponivel  e ON d.product_id = e.product_id
    JOIN ruptura_rj.bronze_parts p ON d.product_id = p.product_id
)
SELECT * FROM analise_ruptura;

-- Resumo de risco
SELECT nivel_risco, COUNT(*) AS produtos, ROUND(SUM(receita_em_risco), 2) AS receita_total_em_risco
FROM ruptura_rj.silver_risco_ruptura
GROUP BY nivel_risco
ORDER BY receita_total_em_risco DESC;

nivel_risco,produtos,receita_total_em_risco
CRITICO,940134,571386.70
ALTO,59583,0.00
MEDIO,282,0.00
BAIXO,1,0.00


In [0]:
-- DIM_PRODUTO
CREATE OR REPLACE TABLE ruptura_rj.dim_produto
USING DELTA
COMMENT 'Dimensão produto com categorização de negócio'
AS
SELECT
    product_id                                       AS sk_produto,
    product_name,
    brand,
    manufacturer,
    product_type,
    retail_price,
    -- Faixa de preço para análise
    CASE
        WHEN retail_price < 500   THEN 'Ticket Baixo'
        WHEN retail_price < 1500  THEN 'Ticket Médio'
        ELSE 'Ticket Alto'
    END AS faixa_preco,
    -- Categoria simplificada (primeiras 2 palavras do tipo)
    TRIM(REGEXP_EXTRACT(product_type, '^(\\w+\\s+\\w+)', 1)) AS categoria
FROM ruptura_rj.bronze_parts;

-- DIM_FORNECEDOR (com risco agregado)
CREATE OR REPLACE TABLE ruptura_rj.dim_fornecedor
USING DELTA
COMMENT 'Dimensão fornecedor com score de confiabilidade'
AS
WITH fornecedor_metricas AS (
    SELECT
        supplier_id,
        ROUND(AVG(delivery_delay_days), 2)                AS media_atraso,
        ROUND(SUM(CASE WHEN is_late_delivery THEN 1.0 ELSE 0 END)
              / COUNT(*) * 100, 1)                        AS pct_entregas_atrasadas
    FROM ruptura_rj.silver_itens
    GROUP BY supplier_id
)
SELECT
    s.supplier_id                                         AS sk_fornecedor,
    s.supplier_name,
    s.account_balance,
    COALESCE(m.media_atraso, 0)                          AS media_atraso_dias,
    COALESCE(m.pct_entregas_atrasadas, 0)                AS pct_entregas_atrasadas,
    CASE
        WHEN COALESCE(m.pct_entregas_atrasadas, 0) < 10  THEN 'Confiável'
        WHEN COALESCE(m.pct_entregas_atrasadas, 0) < 30  THEN 'Atenção'
        ELSE 'Crítico'
    END AS classificacao_fornecedor
FROM ruptura_rj.bronze_supplier s
LEFT JOIN fornecedor_metricas m ON s.supplier_id = m.supplier_id;

-- DIM_TEMPO (calendário a partir dos dados reais)
CREATE OR REPLACE TABLE ruptura_rj.dim_tempo
USING DELTA
COMMENT 'Dimensão tempo gerada a partir das datas de envio'
AS
SELECT DISTINCT
    ship_date                                             AS sk_data,
    ship_year                                             AS ano,
    ship_month                                            AS mes,
    DAY(ship_date)                                        AS dia,
    QUARTER(ship_date)                                    AS trimestre,
    WEEKOFYEAR(ship_date)                                 AS semana_ano,
    DAYOFWEEK(ship_date)                                  AS dia_semana,
    DATE_FORMAT(ship_date, 'MMMM')                       AS nome_mes,
    CASE WHEN DAYOFWEEK(ship_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_fim_de_semana
FROM ruptura_rj.silver_itens
WHERE ship_date IS NOT NULL;

SELECT 'dim_produto'    AS dim, COUNT(*) AS registros FROM ruptura_rj.dim_produto    UNION ALL
SELECT 'dim_fornecedor' AS dim, COUNT(*) AS registros FROM ruptura_rj.dim_fornecedor UNION ALL
SELECT 'dim_tempo'      AS dim, COUNT(*) AS registros FROM ruptura_rj.dim_tempo;

dim,registros
dim_produto,1000000
dim_fornecedor,50000
dim_tempo,2526


In [0]:
-- FATO_VENDAS (tabela central do Star Schema)
CREATE OR REPLACE TABLE ruptura_rj.fato_vendas
USING DELTA
COMMENT 'Fato: vendas com chaves para todas as dimensões'
AS
SELECT
    i.order_id,
    i.product_id       AS sk_produto,
    i.supplier_id      AS sk_fornecedor,
    o.customer_id      AS sk_cliente,
    i.ship_date        AS sk_data,
    i.ship_year,
    i.ship_month,
    i.quantity         AS qtd_vendida,
    i.gross_revenue,
    i.net_revenue,
    i.discount_pct,
    i.delivery_delay_days,
    i.is_late_delivery,
    i.is_returned,
    i.ship_mode
FROM ruptura_rj.silver_itens i
JOIN ruptura_rj.bronze_orders o ON i.order_id = o.order_id;

-- FATO_RUPTURA (tabela analítica de risco)
CREATE OR REPLACE TABLE ruptura_rj.fato_ruptura
USING DELTA
COMMENT 'Fato: produtos em risco de ruptura com receita em perigo'
AS
SELECT
    r.product_id       AS sk_produto,
    r.product_name,
    r.brand,
    r.product_type,
    r.nivel_risco,
    r.score_risco_ruptura,
    r.demanda_total_qty,
    r.estoque_total_disponivel,
    r.cobertura_estoque_dias,
    r.receita_total,
    r.receita_em_risco,
    r.media_atraso_entrega,
    r.qtd_entregas_atrasadas,
    r.num_fornecedores,
    r.ultima_venda
FROM ruptura_rj.silver_risco_ruptura r;

SELECT 'fato_vendas'  AS fato, COUNT(*) AS registros FROM ruptura_rj.fato_vendas  UNION ALL
SELECT 'fato_ruptura' AS fato, COUNT(*) AS registros FROM ruptura_rj.fato_ruptura;

fato,registros
fato_vendas,29999795
fato_ruptura,1000000


In [0]:
-- KPI 1: Receita em risco por nível de criticidade e categoria
CREATE OR REPLACE VIEW ruptura_rj.vw_kpi_receita_em_risco AS
SELECT
    r.nivel_risco,
    p.faixa_preco,
    p.categoria,
    COUNT(DISTINCT r.sk_produto)          AS qtd_produtos,
    ROUND(SUM(r.receita_em_risco), 2)     AS receita_em_risco_total,
    ROUND(AVG(r.score_risco_ruptura), 1)  AS score_medio,
    ROUND(AVG(r.cobertura_estoque_dias))  AS cobertura_media_dias
FROM ruptura_rj.fato_ruptura r
JOIN ruptura_rj.dim_produto p ON r.sk_produto = p.sk_produto
GROUP BY r.nivel_risco, p.faixa_preco, p.categoria;

-- KPI 2: Performance de fornecedores (confiabilidade vs receita gerada)
CREATE OR REPLACE VIEW ruptura_rj.vw_kpi_fornecedor AS
SELECT
    f.supplier_name,
    f.classificacao_fornecedor,
    f.media_atraso_dias,
    f.pct_entregas_atrasadas,
    COUNT(DISTINCT v.order_id)            AS total_pedidos,
    SUM(v.qtd_vendida)                    AS total_qty_entregue,
    ROUND(SUM(v.net_revenue), 2)          AS receita_liquida_gerada,
    SUM(CASE WHEN v.is_returned THEN 1 ELSE 0 END) AS total_devolvidos
FROM ruptura_rj.dim_fornecedor f
JOIN ruptura_rj.fato_vendas v ON f.sk_fornecedor = v.sk_fornecedor
GROUP BY f.supplier_name, f.classificacao_fornecedor,
         f.media_atraso_dias, f.pct_entregas_atrasadas;

-- KPI 3: Tendência mensal de entregas atrasadas vs receita
CREATE OR REPLACE VIEW ruptura_rj.vw_kpi_tendencia_mensal AS
SELECT
    t.ano,
    t.mes,
    t.nome_mes,
    t.trimestre,
    COUNT(DISTINCT v.order_id)                         AS total_pedidos,
    SUM(v.qtd_vendida)                                 AS total_qty,
    ROUND(SUM(v.net_revenue), 2)                       AS receita_liquida,
    SUM(CASE WHEN v.is_late_delivery THEN 1 ELSE 0 END) AS entregas_atrasadas,
    ROUND(
        SUM(CASE WHEN v.is_late_delivery THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1
    )                                                  AS pct_atraso,
    -- Receita perdida estimada (itens devolvidos x preço médio)
    ROUND(
        SUM(CASE WHEN v.is_returned THEN v.net_revenue ELSE 0 END), 2
    )                                                  AS receita_devolvida
FROM ruptura_rj.fato_vendas v
JOIN ruptura_rj.dim_tempo t ON v.sk_data = t.sk_data
GROUP BY t.ano, t.mes, t.nome_mes, t.trimestre
ORDER BY t.ano, t.mes;

SELECT 'vw_kpi_receita_em_risco'  AS view_name, 'OK' AS status UNION ALL
SELECT 'vw_kpi_fornecedor'        AS view_name, 'OK' AS status UNION ALL
SELECT 'vw_kpi_tendencia_mensal'  AS view_name, 'OK' AS status;

view_name,status
vw_kpi_receita_em_risco,OK
vw_kpi_fornecedor,OK
vw_kpi_tendencia_mensal,OK


In [0]:
-- KPI 1 (CORRIGIDO): Top 20 SKUs que mais corroem o caixa
WITH fornecedor_critico_por_produto AS (
    -- Pré-computa o fornecedor mais crítico POR PRODUTO (sem subquery correlacionada)
    SELECT
        ps.product_id,
        FIRST_VALUE(f.supplier_name) OVER (
            PARTITION BY ps.product_id
            ORDER BY f.pct_entregas_atrasadas DESC
        ) AS fornecedor_mais_critico
    FROM ruptura_rj.bronze_partsupp ps
    JOIN ruptura_rj.dim_fornecedor f ON ps.supplier_id = f.sk_fornecedor
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ps.product_id
        ORDER BY f.pct_entregas_atrasadas DESC
    ) = 1
),
base AS (
    -- Join simples: sem subquery correlacionada, sem window ainda
    SELECT
        r.sk_produto,
        r.product_name,
        r.brand,
        r.nivel_risco,
        r.score_risco_ruptura,
        r.receita_total,
        r.receita_em_risco,
        r.cobertura_estoque_dias,
        r.media_atraso_entrega,
        r.num_fornecedores,
        fc.fornecedor_mais_critico
    FROM ruptura_rj.fato_ruptura r
    LEFT JOIN fornecedor_critico_por_produto fc ON r.sk_produto = fc.product_id
    WHERE r.receita_em_risco > 0
),
ranking_impacto AS (
    -- Agora as window functions sem subquery correlacionada
    SELECT
        *,
        ROUND(receita_total / SUM(receita_total) OVER () * 100, 2) AS pct_receita_total,
        ROUND(SUM(receita_em_risco) OVER (
            ORDER BY receita_em_risco DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) AS receita_em_risco_acumulada,
        ROW_NUMBER() OVER (ORDER BY receita_em_risco DESC)         AS ranking
    FROM base
)
SELECT
    ranking,
    product_name,
    brand,
    nivel_risco,
    score_risco_ruptura,
    CONCAT('R$ ', FORMAT_NUMBER(receita_total, 2))              AS receita_historica,
    CONCAT('R$ ', FORMAT_NUMBER(receita_em_risco, 2))           AS receita_em_risco,
    CONCAT(pct_receita_total, '%')                              AS pct_do_total,
    CONCAT(cobertura_estoque_dias, ' dias')                     AS cobertura_estoque,
    CONCAT(media_atraso_entrega, ' dias')                       AS atraso_medio_entrega,
    num_fornecedores,
    fornecedor_mais_critico,
    CONCAT('R$ ', FORMAT_NUMBER(receita_em_risco_acumulada, 2)) AS perda_acumulada_pareto
FROM ranking_impacto
WHERE ranking <= 20
ORDER BY ranking;

ranking,product_name,brand,nivel_risco,score_risco_ruptura,receita_historica,receita_em_risco,pct_do_total,cobertura_estoque,atraso_medio_entrega,num_fornecedores,fornecedor_mais_critico,perda_acumulada_pareto
1,yellow blush blanched papaya white,Brand#45,CRITICO,90.0,"R$ 1,897,053.25","R$ 297,013.39",66.92%,307.9 dias,15.57 dias,4,Supplier#000047989,"R$ 297,013.39"
2,dim royal midnight navy blanched,Brand#55,CRITICO,73.9,"R$ 937,731.58","R$ 274,373.31",33.08%,258.2 dias,4.62 dias,4,Supplier#000000125,"R$ 571,386.70"


Databricks visualization. Run in Databricks to view.

In [0]:
WITH fornecedor_analise AS (
    SELECT
        f.supplier_name,
        f.classificacao_fornecedor,
        f.media_atraso_dias,
        f.pct_entregas_atrasadas,
        COUNT(DISTINCT v.order_id)                             AS total_pedidos,
        ROUND(SUM(v.net_revenue), 2)                          AS receita_liquida,
        ROUND(SUM(v.net_revenue) / SUM(SUM(v.net_revenue))
              OVER () * 100, 2)                               AS pct_receita_total,
        COUNT(DISTINCT v.sk_produto)                          AS skus_atendidos,
        SUM(CASE WHEN v.is_late_delivery THEN 1 ELSE 0 END)  AS total_atrasos,
        SUM(CASE WHEN v.is_returned      THEN 1 ELSE 0 END)  AS total_devolvidos,
        ROUND(SUM(CASE WHEN v.is_returned THEN v.net_revenue ELSE 0 END), 2) AS receita_devolvida,
        -- Ranking de confiabilidade dentro da classificação
        RANK() OVER (
            PARTITION BY f.classificacao_fornecedor
            ORDER BY f.pct_entregas_atrasadas ASC
        ) AS rank_confiabilidade_segmento
    FROM ruptura_rj.dim_fornecedor f
    JOIN ruptura_rj.fato_vendas v ON f.sk_fornecedor = v.sk_fornecedor
    GROUP BY f.supplier_name, f.classificacao_fornecedor,
             f.media_atraso_dias, f.pct_entregas_atrasadas
),
quartis AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY receita_liquida DESC)       AS quartil_receita,
        NTILE(4) OVER (ORDER BY pct_entregas_atrasadas ASC) AS quartil_confiabilidade
    FROM fornecedor_analise
)
SELECT
    supplier_name                         AS fornecedor,
    classificacao_fornecedor,
    CONCAT(pct_entregas_atrasadas, '%')   AS pct_atrasos,
    CONCAT(media_atraso_dias, ' dias')    AS atraso_medio,
    CONCAT('R$ ', FORMAT_NUMBER(receita_liquida, 2)) AS receita_liquida,
    CONCAT(pct_receita_total, '%')        AS pct_do_faturamento,
    skus_atendidos                        AS qtd_skus,
    total_devolvidos,
    CONCAT('R$ ', FORMAT_NUMBER(receita_devolvida, 2)) AS receita_devolvida,
    -- Quadrante estratégico
    CASE
        WHEN quartil_receita = 1 AND quartil_confiabilidade = 1 THEN '🟢 Estratégico'
        WHEN quartil_receita = 1 AND quartil_confiabilidade > 2 THEN '🔴 Risco Crítico'
        WHEN quartil_receita > 2 AND quartil_confiabilidade = 1 THEN '🔵 Parceiro em Crescimento'
        ELSE '🟡 Monitorar'
    END AS quadrante_estrategico
FROM quartis
ORDER BY receita_liquida DESC
LIMIT 30;

fornecedor,classificacao_fornecedor,pct_atrasos,atraso_medio,receita_liquida,pct_do_faturamento,qtd_skus,total_devolvidos,receita_devolvida,quadrante_estrategico
Supplier#000007491,Crítico,61.8%,16.87 dias,"R$ 30,473,160.16",0.00%,80,169,"R$ 7,953,894.69",🟢 Estratégico
Supplier#000024503,Crítico,65.1%,17.23 dias,"R$ 30,255,598.98",0.00%,80,162,"R$ 6,835,732.38",🔴 Risco Crítico
Supplier#000041492,Crítico,61.6%,14.66 dias,"R$ 30,163,712.44",0.00%,80,184,"R$ 8,470,276.28",🟢 Estratégico
Supplier#000022496,Crítico,64.3%,17.41 dias,"R$ 30,153,199.55",0.00%,80,177,"R$ 7,607,877.08",🔴 Risco Crítico
Supplier#000024983,Crítico,63.4%,16.08 dias,"R$ 30,077,203.91",0.00%,80,148,"R$ 6,745,869.16",🔴 Risco Crítico
Supplier#000003457,Crítico,62.4%,14.31 dias,"R$ 29,994,868.38",0.00%,80,163,"R$ 6,654,711.95",🟡 Monitorar
Supplier#000032987,Crítico,63.1%,15.54 dias,"R$ 29,977,487.17",0.00%,80,162,"R$ 6,920,370.98",🟡 Monitorar
Supplier#000011959,Crítico,62.3%,14.48 dias,"R$ 29,892,448.51",0.00%,80,167,"R$ 7,833,082.70",🟡 Monitorar
Supplier#000033997,Crítico,60.8%,15.69 dias,"R$ 29,826,130.19",0.00%,80,137,"R$ 5,962,062.94",🟢 Estratégico
Supplier#000013970,Crítico,62.9%,16.9 dias,"R$ 29,622,804.51",0.00%,80,167,"R$ 7,746,262.69",🟡 Monitorar


Databricks visualization. Run in Databricks to view.

In [0]:
WITH tendencia AS (
    SELECT
        ano,
        mes,
        nome_mes,
        trimestre,
        receita_liquida,
        pct_atraso,
        entregas_atrasadas,
        receita_devolvida,
        -- Variação MoM (mês a mês)
        LAG(receita_liquida) OVER (ORDER BY ano, mes)  AS receita_mes_anterior,
        LAG(pct_atraso)      OVER (ORDER BY ano, mes)  AS pct_atraso_mes_anterior,
        -- Média móvel 3 meses (suaviza sazonalidade)
        ROUND(AVG(receita_liquida) OVER (
            ORDER BY ano, mes
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS media_movel_3m_receita,
        ROUND(AVG(pct_atraso) OVER (
            ORDER BY ano, mes
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS media_movel_3m_atraso,
        -- Rank do pior mês de atraso por ano
        RANK() OVER (PARTITION BY ano ORDER BY pct_atraso DESC) AS rank_pior_mes_ano
    FROM ruptura_rj.vw_kpi_tendencia_mensal
)
SELECT
    ano,
    mes,
    nome_mes,
    CONCAT('T', trimestre)                                         AS trimestre,
    CONCAT('R$ ', FORMAT_NUMBER(receita_liquida, 2))              AS receita_mes,
    CONCAT('R$ ', FORMAT_NUMBER(media_movel_3m_receita, 2))       AS media_movel_3m,
    CONCAT(pct_atraso, '%')                                        AS pct_entregas_atrasadas,
    CONCAT(media_movel_3m_atraso, '%')                            AS tendencia_atraso_3m,
    -- Variação percentual MoM
    CASE
        WHEN receita_mes_anterior IS NOT NULL
        THEN CONCAT(
            ROUND((receita_liquida - receita_mes_anterior) / receita_mes_anterior * 100, 1),
            '%'
        )
        ELSE 'N/A'
    END AS variacao_mom_receita,
    CASE
        WHEN pct_atraso_mes_anterior IS NOT NULL
        THEN CONCAT(
            ROUND(pct_atraso - pct_atraso_mes_anterior, 1),
            'pp'
        )
        ELSE 'N/A'
    END AS variacao_mom_atraso,
    CONCAT('R$ ', FORMAT_NUMBER(receita_devolvida, 2))            AS receita_perdida_devolucao,
    CASE WHEN rank_pior_mes_ano = 1 THEN '⚠️ Pior mês do ano' ELSE '' END AS alerta
FROM tendencia
ORDER BY ano, mes;

ano,mes,nome_mes,trimestre,receita_mes,media_movel_3m,pct_entregas_atrasadas,tendencia_atraso_3m,variacao_mom_receita,variacao_mom_atraso,receita_perdida_devolucao,alerta
1992,1,January,T1,"R$ 1,813,000,113.44","R$ 1,813,000,113.44",4.9%,4.9%,N/A,N/A,"R$ 905,984,775.94",
1992,2,February,T1,"R$ 5,078,696,027.26","R$ 3,445,848,070.35",20.1%,12.5%,180.1%,15.2pp,"R$ 2,548,222,738.09",
1992,3,March,T1,"R$ 9,062,735,243.65","R$ 5,318,143,794.78",41.7%,22.2%,78.4%,21.6pp,"R$ 4,538,951,353.62",
1992,4,April,T2,"R$ 12,286,098,618.98","R$ 8,809,176,629.96",58.0%,39.9%,35.6%,16.3pp,"R$ 6,143,345,917.40",
1992,5,May,T2,"R$ 14,557,548,441.82","R$ 11,968,794,101.48",63.3%,54.3%,18.5%,5.3pp,"R$ 7,273,701,983.98",⚠️ Pior mês do ano
1992,6,June,T2,"R$ 14,134,080,734.82","R$ 13,659,242,598.54",63.3%,61.5%,-2.9%,0.0pp,"R$ 7,073,081,680.52",⚠️ Pior mês do ano
1992,7,July,T3,"R$ 14,556,697,336.14","R$ 14,416,108,837.59",63.2%,63.3%,3.0%,-0.1pp,"R$ 7,276,147,319.21",
1992,8,August,T3,"R$ 14,595,350,610.36","R$ 14,428,709,560.44",63.1%,63.2%,0.3%,-0.1pp,"R$ 7,286,055,256.51",
1992,9,September,T3,"R$ 14,104,487,973.04","R$ 14,418,845,306.51",63.3%,63.2%,-3.4%,0.2pp,"R$ 7,076,927,775.44",⚠️ Pior mês do ano
1992,10,October,T4,"R$ 14,578,264,665.57","R$ 14,426,034,416.32",63.2%,63.2%,3.4%,-0.1pp,"R$ 7,297,803,511.76",


Databricks visualization. Run in Databricks to view.